# Experiment 14 — Does random sampling break at very low data fractions where stratified does not? (2026-07-18)

Direct test of the theory from "Why stratified sampling has actually been winning" (EXPERIMENTS.md): a side-analysis (not a search run) measured that at 0.1%-0.2% starting fractions, `stratified`'s low-discrepancy pick order guarantees near-total store coverage (601/601 stores at 0.2%) while true random sampling's coverage degrades by luck (448.1/601 average across 20 seeds at 0.2%, worst seed 433). That side-analysis predicted, but did not measure, that an actual search using `subsample='random'` would start to lose to `stratified` below ~0.5% - no search had ever actually been run with `subsample='random'` on this dataset. This experiment runs that search.

**Primary arm**: one Patient/Eager round, `subsample='stratified'`, zones `[0.2%, 0.3%, 0.5%, 100%]` - pushing one rung below Experiment 11's 0.25% floor, into the range the coverage analysis flagged as where `stratified` should decisively separate from random luck.

**Comparison arm**: the same zones ladder and the same `patient` policy, but `subsample='random'` instead of `'stratified'` - single seed (`random_state=0`, this project's standard convention), same as every other P/E round logged so far. If the theory holds, this arm should show worse search behavior (more evaluations to converge, a worse optimum, or both) than the matched `stratified` arm at the same starting fraction.

In [1]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

SplitPoint:  418416


C:\Users\Home\AppData\Local\Temp\ipykernel_15588\4255920482.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


Training Set shape (418416, 31)
Validation Set shape (104605, 31)
Feature Columns:
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']
Pipeline done in 12.3 s


In [2]:
# --- common setup: grid, CV, zones ladder [0.2%, 0.3%, 0.5%, 100%] ---
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from bayes_halving_search_cv import PatternSearchCV

X, y = trainDataset_X, trainDataset_y
N_features = X.shape[1]
ZONES = [0.002, 0.003, 0.005, 1.0]

def make_clf():
    return ExtraTreesRegressor(n_estimators=240, max_features=max(1, N_features - 15),
                               max_depth=16, n_jobs=1, random_state=0)

param_grid = {
    "max_features": [2, 3, 4],
    "n_estimators": list(range(10, 261, 10)),
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}
tscv = TimeSeriesSplit(n_splits=5)

def run_arm(label, contraction, subsample):
    search = PatternSearchCV(make_clf(), param_grid,
                             scoring="neg_mean_absolute_error", cv=tscv,
                             n_jobs=-1, poll="opportunistic",
                             contraction=contraction, data_zones=ZONES,
                             subsample=subsample,
                             random_state=0, verbose=0)
    t0 = time.time()
    search.fit(X, y)
    wall = time.time() - t0
    res = search.cv_results_
    evals = []
    for p, sc, nr, ft in zip(res["params"], res["mean_test_score"],
                             res["n_resources"], res["mean_fit_time"]):
        key = (p["max_features"], p["n_estimators"], p["max_depth"], int(nr))
        evals.append({"key": key, "mae": -sc, "fit_work": float(ft) * 5})
    out = {
        "label": label, "contraction": contraction, "subsample": subsample,
        "wall": wall,
        "n_fits": len(res["params"]),
        "equiv": float(np.sum(np.asarray(res["n_resources"]) / len(y))),
        "best": search.best_params_, "best_mae": -search.best_score_,
        "zones_used": sorted(set(int(v) for v in res["n_resources"])),
        "fit_work_total": sum(e["fit_work"] for e in evals),
        "evals": evals,
    }
    print(f"{label:28s}: {out['n_fits']} evals, {out['equiv']:.2f} equiv, "
         f"{wall:.1f} s wall, best {out['best']} MAE {out['best_mae']:.3f}")
    return out

In [3]:
P_strat = run_arm("PATIENT / stratified", "patient", "stratified")

PATIENT / stratified        : 31 evals, 5.07 equiv, 1168.0 s wall, best {'max_features': 4, 'n_estimators': 130, 'max_depth': 17} MAE 805.038


In [4]:
E_strat = run_arm("EAGER / stratified", "eager", "stratified")

EAGER / stratified          : 29 evals, 5.06 equiv, 1160.9 s wall, best {'max_features': 4, 'n_estimators': 130, 'max_depth': 17} MAE 805.038


In [5]:
P_rand = run_arm("PATIENT / random", "patient", "random")

PATIENT / random            : 27 evals, 5.07 equiv, 910.0 s wall, best {'max_features': 4, 'n_estimators': 130, 'max_depth': 17} MAE 805.038


In [6]:
# --- three-way comparison, zones [0.2%, 0.3%, 0.5%, 100%] ---
print("=" * 90)
print(f"zones ladder {ZONES}")
print(f"{'':22s}{'PATIENT/stratified':>20s}{'EAGER/stratified':>20s}{'PATIENT/random':>20s}")
arms = [P_strat, E_strat, P_rand]
print(f"{'evaluations':22s}" + "".join(f"{a['n_fits']:>20d}" for a in arms))
print(f"{'full-fit equivalents':22s}" + "".join(f"{a['equiv']:>20.2f}" for a in arms))
print(f"{'wall-clock (s)':22s}" + "".join(f"{a['wall']:>20.1f}" for a in arms))
print(f"{'summed fit work (s)':22s}" + "".join(f"{a['fit_work_total']:>20.1f}" for a in arms))
print(f"{'best point':22s}" + "".join(
    f"{str((a['best']['max_features'], a['best']['n_estimators'], a['best']['max_depth'])):>20s}"
    for a in arms))
print(f"{'best CV MAE':22s}" + "".join(f"{a['best_mae']:>20.3f}" for a in arms))
print(f"{'zones used (rows)':22s}" + "".join(f"{str(a['zones_used']):>20s}" for a in arms))
print()

s_map = {e['key']: e['fit_work'] for e in P_strat['evals']}
r_map = {e['key']: e['fit_work'] for e in P_rand['evals']}
shared = sorted(set(s_map) & set(r_map))
print(f"shared evaluations (PATIENT/stratified vs PATIENT/random): "
     f"{len(shared)} of {P_strat['n_fits']}/{P_rand['n_fits']}")

zones ladder [0.002, 0.003, 0.005, 1.0]
                        PATIENT/stratified    EAGER/stratified      PATIENT/random
evaluations                             31                  29                  27
full-fit equivalents                  5.07                5.06                5.07
wall-clock (s)                      1168.0              1160.9               910.0
summed fit work (s)                 2945.3              2993.7              2290.3
best point                    (4, 130, 17)        (4, 130, 17)        (4, 130, 17)
best CV MAE                        805.038             805.038             805.038
zones used (rows)      [837, 2093, 418416] [837, 2093, 418416] [837, 2093, 418416]

shared evaluations (PATIENT/stratified vs PATIENT/random): 21 of 31/27


## Summary

**The coverage theory did NOT hold up in this test.** All three arms
(PATIENT/stratified, EAGER/stratified, PATIENT/random) converged to the
exact same optimum (4, 130, 17) at the exact same MAE (805.038), with
essentially identical full-fit equivalents (5.06-5.07). `PATIENT/random`
actually used *fewer* evaluations (27 vs 31) and less wall-clock than its
matched `PATIENT/stratified` arm - the opposite direction from what the
store-coverage side-analysis predicted (601/601 stores for `stratified` vs
448.1/601 average for `random` at this fraction).

This is a single-seed result (`random_state=0`) and does not overturn the
coverage side-analysis's real, measured gap - it shows that gap didn't
translate into a worse search *outcome* for this specific seed, grid, and
search algorithm. Plausible reasons: (1) seed 0 may have drawn a lucky
0.2% sample: the coverage numbers had real seed-to-seed variance; (2)
PatternSearchCV's mesh contraction and ratcheted zone growth mean a noisy
small-sample signal only has to be roughly right, not precisely ranked,
before the search self-corrects at a larger zone; (3) ExtraTreesRegressor's
own internal randomization already introduces comparable ranking noise.

Logged as Experiment 14 in EXPERIMENTS.md, with the multi-seed follow-up
this result calls for noted as still open.